# 1. Data Collection — SpaceX REST API

Collect Falcon 9 launch data using the SpaceX public REST API.

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1.1 Helper Functions

In [ ]:
def getBoosterVersion(data):
    """Get booster version from rocket ID."""
    for x in data['rocket']:
        response = requests.get(f'https://api.spacexdata.com/v4/rockets/{x}').json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    """Get launch site name and coordinates."""
    for x in data['launchpad']:
        response = requests.get(f'https://api.spacexdata.com/v4/launchpads/{x}').json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    """Get payload mass and orbit."""
    for load in data['payloads']:
        response = requests.get(f'https://api.spacexdata.com/v4/payloads/{load}').json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

def getCoreData(data):
    """Get core/booster landing data."""
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get(f'https://api.spacexdata.com/v4/cores/{core["core"]}').json()
            Block.append(response.get('block', None))
            ReusedCount.append(response.get('reuse_count', None))
            Serial.append(response.get('serial', None))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core.get('landing_success', None)) + ' ' + str(core.get('landing_type', None)))
        Flights.append(core.get('flight', None))
        GridFins.append(core.get('gridfins', None))
        Reused.append(core.get('reused', None))
        Legs.append(core.get('legs', None))
        LandingPad.append(core.get('landpad', None))


## 1.2 Fetch All Falcon 9 Launches

In [ ]:
# Fetch all launches
spacex_url = 'https://api.spacexdata.com/v4/launches/past'
response = requests.get(spacex_url)
print('Status code:', response.status_code)
data = pd.json_normalize(response.json())
print('Total launches:', len(data))
data.head(3)


In [ ]:
# Filter Falcon 9 only
data_falcon9 = data[data['rocket'] == '5e9d0d95aca009fea7ea0097']
data_falcon9 = data_falcon9.reset_index(drop=True)
print('Falcon 9 launches:', len(data_falcon9))


## 1.3 Extract All Fields

In [ ]:
# Initialise lists
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []
Longitude, Latitude = [], []

# Populate lists via API calls
getBoosterVersion(data_falcon9)
getLaunchSite(data_falcon9)
getPayloadData(data_falcon9)
getCoreData(data_falcon9)

print('Data collected for', len(BoosterVersion), 'launches')


## 1.4 Build DataFrame

In [ ]:
launch_dict = {
    'FlightNumber':   list(data_falcon9['flight_number']),
    'Date':           list(data_falcon9['date_utc']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass':    PayloadMass,
    'Orbit':          Orbit,
    'LaunchSite':     LaunchSite,
    'Outcome':        Outcome,
    'Flights':        Flights,
    'GridFins':       GridFins,
    'Reused':         Reused,
    'Legs':           Legs,
    'LandingPad':     LandingPad,
    'Block':          Block,
    'ReusedCount':    ReusedCount,
    'Serial':         Serial,
    'Longitude':      Longitude,
    'Latitude':       Latitude,
}

df = pd.DataFrame(launch_dict)
print(df.shape)
df.head()


## 1.5 Filter to Falcon 9 v1.1+ and Save

In [ ]:
# Keep only Falcon 9 (exclude FH / early test)
df = df[~df['BoosterVersion'].str.contains('Falcon 1')]
df.loc[:,'FlightNumber'] = list(range(1, df.shape[0]+1))

# Handle missing PayloadMass
df['PayloadMass'] = df['PayloadMass'].astype(float)
df['PayloadMass'].fillna(df['PayloadMass'].mean(), inplace=True)

print('Final shape:', df.shape)
print('Missing values:\n', df.isnull().sum())
df.to_csv('../data/dataset_part_1.csv', index=False)
print('Saved to dataset_part_1.csv')
